# Cypher Query Examples — `one_rainbow` KG

Quick reference notebook with common Cypher query patterns against the **`one_rainbow`** knowledge graph (Rainbow opportunity reviews).

## Setup
Run the cell below to initialise the environment and connect to the KG.


In [1]:
from genai_tk.utils.config_mngr import global_config
from rich.console import Console
from rich.table import Table

from genai_graph.kg.backend import create_backend_from_config
from genai_graph.kg.embeddings_handler import EmbeddingsHandler

console = Console()

# Connect to the one_rainbow KG output folder
backend = create_backend_from_config("default", "one_rainbow")


def run_query(query: str, title: str = "Results", params: dict | None = None):
    """Execute a Cypher query and display results as a Rich table."""
    try:
        result = backend.execute(query, params or {})
        df = result.get_as_df()

        table = Table(title=f"{title} ({len(df)} rows)")
        for col in df.columns:
            table.add_column(str(col), style="cyan")
        for _, row in df.iterrows():
            table.add_row(*[str(val) for val in row])

        console.print(table)
        return df
    except Exception as e:
        console.print(f"[red]Error: {e}[/red]")
        return None


print("✅ Connected to one_rainbow KG. Ready to run queries!")

✅ Connected to one_rainbow KG. Ready to run queries!


## Basic Queries


In [2]:
# Count nodes by type
query = "MATCH (n) RETURN labels(n)[0] as type, count(n) as count ORDER BY count DESC"
run_query(query, "Node Counts by Type")

 Node Counts by 
 Type (1 rows)  
┏━━━━━━┳━━━━━━━┓
┃ type ┃ count ┃
┡━━━━━━╇━━━━━━━┩
│      │ 33    │
└──────┴───────┘

,type,count
0,,33


In [3]:
# List relationship types
query = "MATCH ()-[r]->() RETURN type(r) as relationship, count(*) as count ORDER BY count DESC"
run_query(query, "Relationship Types")

Error: Catalog exception: function TYPE does not exist.

In [4]:
# Most recent reviewed opportunities
run_query(
    "MATCH (n:ReviewedOpportunity) RETURN n.name, n.start_date ORDER BY n.start_date DESC LIMIT 10",
    "Recent Reviewed Opportunities",
)

    Recent Reviewed Opportunities (1 rows)     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ n.name                       ┃ n.start_date ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ 2019-04-01   │
└──────────────────────────────┴──────────────┘

,n.name,n.start_date
0,Review:9000559500:2019-04-01,2019-04-01


## Relationship Queries


In [5]:
# Opportunities with competitors
query = """
MATCH (ro:ReviewedOpportunity)-[:HAS_COMPETITOR]->(c:Competitor)
RETURN ro.name as opportunity, c.name as competitor
ORDER BY ro.name
LIMIT 10
"""
run_query(query, "Opportunities with Competitors")

   Opportunities with Competitors (3 rows)   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ opportunity                  ┃ competitor ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ Capgemini  │
│ Review:9000559500:2019-04-01 │ Thales     │
│ Review:9000559500:2019-04-01 │ Other      │
└──────────────────────────────┴────────────┘

,opportunity,competitor
0,Review:9000559500:2019-04-01,Capgemini
1,Review:9000559500:2019-04-01,Thales
2,Review:9000559500:2019-04-01,Other


In [6]:
# Reviews → Opportunity → Customer
run_query(
    """
    MATCH (ro:ReviewedOpportunity)-[:REVIEWS]->(opp:Opportunity)-[:HAS_CUSTOMER]->(cust:Customer)
    RETURN ro.name AS review, opp.name AS opportunity, cust.name AS customer
    ORDER BY ro.name
    LIMIT 10
    """,
    "Reviews → Opportunity → Customer",
)

                      Reviews → Opportunity → Customer (1 rows)                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ review                       ┃ opportunity                             ┃ customer ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ CNES TMA VENUS VIP, PEPS, THEIA MUSCATE │ CNES     │
└──────────────────────────────┴─────────────────────────────────────────┴──────────┘

,review,opportunity,customer
0,Review:9000559500:2019-04-01,"CNES TMA VENUS VIP, PEPS, THEIA MUSCATE",CNES


In [7]:
# Count competitors per opportunity
query = """
MATCH (ro:ReviewedOpportunity)-[:HAS_COMPETITOR]->(c:Competitor)
RETURN ro.name as opportunity, count(c) as competitor_count
ORDER BY competitor_count DESC
LIMIT 10
"""
run_query(query, "Competitors per Opportunity")

       Competitors per Opportunity (1 rows)        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ opportunity                  ┃ competitor_count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ 3                │
└──────────────────────────────┴──────────────────┘

,opportunity,competitor_count
0,Review:9000559500:2019-04-01,3


## Statistical Queries


In [8]:
# Financial statistics
query = """
MATCH (n:ReviewedOpportunity)
RETURN 
  count(n) as total_count,
  avg(n.financials.tcv) as avg_tcv,
  max(n.financials.tcv) as max_tcv,
  min(n.financials.tcv) as min_tcv
"""
run_query(query, "Financial Statistics")

           Financial Statistics (1 rows)           
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ total_count ┃ avg_tcv   ┃ max_tcv   ┃ min_tcv   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━┩
│ 1.0         │ 2380000.0 │ 2380000.0 │ 2380000.0 │
└─────────────┴───────────┴───────────┴───────────┘

,total_count,avg_tcv,max_tcv,min_tcv
0,1,2380000.0,2380000.0,2380000.0


In [9]:
# Risk category distribution
run_query(
    """
    MATCH (r:RiskAnalysis)
    RETURN r.risk_category AS category, count(r) AS count
    ORDER BY count DESC
    """,
    "Risk Categories",
)

   Risk Categories (5 rows)    
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ category            ┃ count ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ SLAComplianceRisk   │ 1     │
│ nan                 │ 1     │
│ DeliveryCostOverrun │ 1     │
│ ScopeManagementRisk │ 1     │
│ SkillRetentionRisk  │ 1     │
└─────────────────────┴───────┘

,category,count
0,SLAComplianceRisk,1
1,NaN,1
2,DeliveryCostOverrun,1
3,ScopeManagementRisk,1
4,SkillRetentionRisk,1


## Embedding / Vector Search

Kuzu's `vector` extension exposes `CALL QUERY_VECTOR_INDEX(...)` as a native
Cypher clause. The result binds `node` and `distance` variables that can be piped
directly into graph traversals — no Python glue code required.

```cypher
CALL QUERY_VECTOR_INDEX('<Table>', '<index_name>', $query_vector, <k>)
WITH node AS n, distance
MATCH (n)-[:REL]->()
RETURN n.name, distance
ORDER BY distance;
```

The rainbow schema has the following vector indexes (built at KG build time):

| Node | Indexed field | Index name |
|---|---|---|
| `Opportunity` | `name` | `name_index` |
| `Customer` | `name` | `name_index` |
| `RiskAnalysis` | `risk_description` | `risk_description_index` |
| `TechnicalApproach` | `architecture` | `architecture_index` |
| `TechnicalApproach` | `technical_stack` | `technical_stack_index` |


In [10]:
# Load the vector extension and initialise the embeddings handler
backend.ensure_vector_extension()

cfg = global_config()
embeddings_id = cfg.get_str("kg_build.embeddings.default")
print(f"Embeddings model: {embeddings_id}")

emb_handler = EmbeddingsHandler(embeddings_id=embeddings_id)

# Verify indexes exist
df_idx = backend.execute("CALL SHOW_INDEXES() RETURN *;").get_as_df()
console.print(df_idx.to_string(index=False))

2026-02-26 21:11:08.587 | DEBUG    | genai_tk.core.embeddings_factory:get_embeddings:571 - get embeddings: 'qwen3_06b@deepinfra' -cache: True
2026-02-26 21:11:08.689 | DEBUG    | genai_graph.kg.embeddings_handler:__init__:49 - EmbeddingsHandler initialized with model: qwen3_06b@deepinfra


Embeddings model: qwen3_06b@deepinfra


table_name             index_name index_type               property_names  extension_loaded                 
index_definition
TechnicalApproach  technical_stack_index       HNSW                True CALL 
CREATE_VECTOR_INDEX('TechnicalApproach', 'technical_stack_index', 'technical_stack_embedding', mu := 30, ml := 60, 
pu := 0.050000, metric := 'cosine', alpha := 1.100000, efc := 200);
TechnicalApproach     architecture_index       HNSW                   True       CALL 
CREATE_VECTOR_INDEX('TechnicalApproach', 'architecture_index', 'architecture_embedding', mu := 30, ml := 60, pu := 
0.050000, metric := 'cosine', alpha := 1.100000, efc := 200);
     RiskAnalysis risk_description_index       HNSW               True    CALL CREATE_VECTOR_INDEX('RiskAnalysis', 
'risk_description_index', 'risk_description_embedding', mu := 30, ml := 60, pu := 0.050000, metric := 'cosine', 
alpha := 1.100000, efc := 200);
         Customer             name_index       HNSW                           True                                
CALL CREATE_VECTOR_INDEX('Customer', 'name_index', 'name_embedding', mu := 30, ml := 60, pu := 0.050000, metric := 
'cosine', alpha := 1.100000, efc := 200);
      Opportunity             name_index       HNSW                           True                             CALL
CREATE_VECTOR_INDEX('Opportunity', 'name_index', 'name_embedding', mu := 30, ml := 60, pu := 0.050000, metric := 
'cosine', alpha := 1.100000, efc := 200);

In [11]:
# Check embedding coverage in the production KG
for table in ["Opportunity", "Customer", "RiskAnalysis", "TechnicalApproach"]:
    try:
        r = backend.execute(f"MATCH (n:{table}) RETURN count(*) AS total, count(n.name_embedding) AS with_emb")
        df = r.get_as_df()
        print(f"{table}: {df.to_string(index=False)}")
    except Exception as e:
        # Try risk_description / architecture / technical_stack for other tables
        try:
            fields_map = {
                "RiskAnalysis": "risk_description_embedding",
                "TechnicalApproach": "architecture_embedding",
            }
            emb_field = fields_map.get(table, "name_embedding")
            r = backend.execute(f"MATCH (n:{table}) RETURN count(*) AS total, count(n.{emb_field}) AS with_emb")
            df = r.get_as_df()
            print(f"{table}.{emb_field}: {df.to_string(index=False)}")
        except Exception as e2:
            print(f"{table}: error — {e2}")

Opportunity:  total  with_emb
     1         1
Customer:  total  with_emb
     1         1
RiskAnalysis.risk_description_embedding:  total  with_emb
     5         0
TechnicalApproach.architecture_embedding:  total  with_emb
     2         2


In [12]:
print("RiskAnalysis schema:")
print(backend.execute("CALL table_info('RiskAnalysis') RETURN *;").get_as_df().to_string(index=False))
print()
df_risk = backend.execute("MATCH (n:RiskAnalysis) RETURN n.name").get_as_df()
print(df_risk.to_string())

RiskAnalysis schema:
 property id                       name        type default expression  primary key
           0                         id      STRING               NULL         True
           1                       name      STRING               NULL        False
           2             _original_name      STRING               NULL        False
           3                _created_at      STRING               NULL        False
           4                _updated_at      STRING               NULL        False
           5              risk_category      STRING               NULL        False
           6 risk_description_embedding FLOAT[1024]               NULL        False

                n.name
0    SLAComplianceRisk
1  DeliveryCostOverrun
2  ScopeManagementRisk
3   SkillRetentionRisk
4                 None


In [13]:
import pathlib
import tempfile

import kuzu

# ── Build a tiny demo database that mirrors the rainbow schema ──────────────
# Kuzu 0.11.3 does not allow SET on an indexed FLOAT[N] column, so we populate
# embeddings during node creation (as the KG build pipeline does).
# This demo DB lets us exercise CALL QUERY_VECTOR_INDEX immediately.

DEMO_DIR = pathlib.Path(tempfile.mkdtemp(prefix="kuzu_vec_demo_"))
DEMO_PATH = DEMO_DIR / "demo.db"
demo_db = kuzu.Database(str(DEMO_PATH))
demo_conn = kuzu.Connection(demo_db)

DIM = len(emb_handler.compute_embeddings("warmup"))  # e.g. 1024

demo_conn.execute("INSTALL VECTOR; LOAD VECTOR;")
demo_conn.execute(f"""
    CREATE NODE TABLE Opportunity(
        name STRING PRIMARY KEY,
        name_embedding FLOAT[{DIM}]
    );
""")
demo_conn.execute(f"""
    CREATE NODE TABLE RiskAnalysis(
        name STRING PRIMARY KEY,
        risk_description STRING,
        risk_description_embedding FLOAT[{DIM}]
    );
""")
demo_conn.execute("""
    CREATE REL TABLE HAS_RISK(FROM Opportunity TO RiskAnalysis);
""")

# ── Sample data ─────────────────────────────────────────────────────────────
opportunities = [
    "CNES TMA VENUS infrastructure modernisation",
    "Cloud-native migration for satellite data platform",
    "Cybersecurity compliance programme — aerospace",
    "Supply chain ERP consolidation",
]
risks = [
    (
        "Integration complexity with legacy systems",
        "Multiple legacy interfaces require bespoke adapters, increasing schedule risk",
    ),
    ("Cloud vendor lock-in", "Heavy reliance on a single cloud provider limits future flexibility"),
    ("Regulatory certification delay", "ESA compliance approval may extend by 3–6 months due to new security rules"),
    ("Supply chain shortage", "Key hardware components have 6-month lead times due to global shortages"),
    ("Key personnel turnover", "Senior architects may leave mid-project, causing knowledge loss"),
]

for opp in opportunities:
    emb = emb_handler.compute_embeddings(opp)
    demo_conn.execute(
        "CREATE (:Opportunity {name: $n, name_embedding: $e})",
        {"n": opp, "e": emb},
    )

for risk_name, risk_desc in risks:
    emb = emb_handler.compute_embeddings(risk_desc)
    demo_conn.execute(
        "CREATE (:RiskAnalysis {name: $n, risk_description: $d, risk_description_embedding: $e})",
        {"n": risk_name, "d": risk_desc, "e": emb},
    )

# Link each opportunity to 2 risks (round-robin)
for i, opp in enumerate(opportunities):
    demo_conn.execute(
        "MATCH (o:Opportunity {name: $o}) MATCH (r:RiskAnalysis {name: $r}) CREATE (o)-[:HAS_RISK]->(r)",
        {"o": opp, "r": risks[i % len(risks)][0]},
    )
    demo_conn.execute(
        "MATCH (o:Opportunity {name: $o}) MATCH (r:RiskAnalysis {name: $r}) CREATE (o)-[:HAS_RISK]->(r)",
        {"o": opp, "r": risks[(i + 1) % len(risks)][0]},
    )

# ── Vector indexes ───────────────────────────────────────────────────────────
demo_conn.execute(
    "CALL CREATE_VECTOR_INDEX('Opportunity',  'name_index',             'name_embedding',             metric := 'cosine');"
)
demo_conn.execute(
    "CALL CREATE_VECTOR_INDEX('RiskAnalysis', 'risk_description_index', 'risk_description_embedding', metric := 'cosine');"
)

print(f"✅ Demo DB ready — {DIM}-dim embeddings, 2 vector indexes")

2026-02-26 21:11:10.118 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (6 chars) -> 1024 dims
2026-02-26 21:11:11.122 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (43 chars) -> 1024 dims
2026-02-26 21:11:12.020 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (50 chars) -> 1024 dims
2026-02-26 21:11:12.801 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (46 chars) -> 1024 dims
2026-02-26 21:11:14.054 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (30 chars) -> 1024 dims
2026-02-26 21:11:14.831 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (77 chars) -> 1024 dims
2026-02-26 21:11:15.598 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for t

✅ Demo DB ready — 1024-dim embeddings, 2 vector indexes


In [14]:
# Simple vector search — pure Cypher CALL QUERY_VECTOR_INDEX
# Returns the k nearest Opportunity nodes sorted by cosine distance
query_text = "cloud infrastructure modernisation"
query_vector = emb_handler.compute_embeddings(query_text)

df = demo_conn.execute(
    """
    CALL QUERY_VECTOR_INDEX('Opportunity', 'name_index', $qv, 3)
    RETURN node.name AS opportunity, round(distance, 4) AS distance
    ORDER BY distance;
    """,
    {"qv": query_vector},
).get_as_df()

console.print(f"\n[bold]Top-3 Opportunities ≈[/bold] '{query_text}'")
console.print(df.to_string(index=False))

2026-02-26 21:11:18.871 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (34 chars) -> 1024 dims


Top-3 Opportunities ≈ 'cloud infrastructure modernisation'

opportunity  distance
Cloud-native migration for satellite data platform    0.2628
       CNES TMA VENUS infrastructure modernisation    0.4125
                    Supply chain ERP consolidation    0.4186

In [15]:
# Vector search + graph traversal in one Cypher query
# CALL QUERY_VECTOR_INDEX binds `node` and `distance`, which can be piped straight
# into a MATCH for further graph traversal — no Python loop needed.
query_text = "supply chain delivery delay"
query_vector = emb_handler.compute_embeddings(query_text)

df = demo_conn.execute(
    """
    CALL QUERY_VECTOR_INDEX('RiskAnalysis', 'risk_description_index', $qv, 3)
    WITH node AS r, distance
    MATCH (opp:Opportunity)-[:HAS_RISK]->(r)
    RETURN opp.name           AS opportunity,
           r.name             AS risk,
           r.risk_description AS description,
           round(distance, 4) AS distance
    ORDER BY distance;
    """,
    {"qv": query_vector},
).get_as_df()

console.print(f"\n[bold]Risks ≈[/bold] '{query_text}' + linked opportunity")
console.print(df.to_string(index=False))

2026-02-26 21:11:19.655 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (27 chars) -> 1024 dims


Risks ≈ 'supply chain delivery delay' + linked opportunity

opportunity                                       risk                      
description  distance
                    Supply chain ERP consolidation                      Supply chain shortage       Key hardware 
components have 6-month lead times due to global shortages    0.3776
    Cybersecurity compliance programme — aerospace                      Supply chain shortage       Key hardware 
components have 6-month lead times due to global shortages    0.3776
Cloud-native migration for satellite data platform                       Cloud vendor lock-in           Heavy 
reliance on a single cloud provider limits future flexibility    0.5062
       CNES TMA VENUS infrastructure modernisation                       Cloud vendor lock-in           Heavy 
reliance on a single cloud provider limits future flexibility    0.5062
       CNES TMA VENUS infrastructure modernisation Integration complexity with legacy systems Multiple legacy 
interfaces require bespoke adapters, increasing schedule risk    0.5160

In [16]:
# Filtered vector search via PROJECT_GRAPH_CYPHER
# Restricts the HNSW search space to a matching subset before computing similarity.
# Here: only Opportunity nodes whose name contains "cloud" or "CNES".
query_text = "satellite data cloud platform"
query_vector = emb_handler.compute_embeddings(query_text)

demo_conn.execute("""
    CALL PROJECT_GRAPH_CYPHER(
        'space_opps',
        'MATCH (n:Opportunity)
         WHERE n.name CONTAINS "cloud" OR n.name CONTAINS "CNES"
         RETURN n'
    );
""")

df = demo_conn.execute(
    """
    CALL QUERY_VECTOR_INDEX('space_opps', 'name_index', $qv, 5)
    RETURN node.name AS opportunity, round(distance, 4) AS distance
    ORDER BY distance;
    """,
    {"qv": query_vector},
).get_as_df()

console.print(f"\n[bold]Filtered top-5 (space/cloud subset) ≈[/bold] '{query_text}'")
console.print(df.to_string(index=False))

2026-02-26 21:11:20.420 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (29 chars) -> 1024 dims


Filtered top-5 (space/cloud subset) ≈ 'satellite data cloud platform'

opportunity  distance
CNES TMA VENUS infrastructure modernisation    0.4548

## Quick Reference

### Common Cypher Patterns

**Count nodes:**
```cypher
MATCH (n:Label) RETURN count(n)
```

**Traverse relationships:**
```cypher
MATCH (a)-[:REL_TYPE]->(b) RETURN a, b
```

**Access MAP / STRUCT field:**
```cypher
MATCH (n:ReviewedOpportunity) RETURN n.financials.tcv
```

**List vector indexes:**
```cypher
CALL SHOW_INDEXES() RETURN *;
```

### Vector Search Patterns

**Simple similarity search:**
```cypher
CALL QUERY_VECTOR_INDEX('<Table>', '<index>', $query_vector, <k>)
RETURN node.name, round(distance, 4) AS distance
ORDER BY distance;
```

**Vector search → graph traversal (single query):**
```cypher
CALL QUERY_VECTOR_INDEX('RiskAnalysis', 'risk_description_index', $query_vector, 10)
WITH node AS r, distance
MATCH (ro:ReviewedOpportunity)-[:HAS_RISK]->(r)
RETURN ro.name AS opportunity, r.name AS risk, distance
ORDER BY distance;
```

**Filtered vector search (PROJECT_GRAPH_CYPHER):**
```python
# Step 1: project a filtered subset
backend.execute("""
    CALL PROJECT_GRAPH_CYPHER(
        'my_subset',
        'MATCH (n:NodeLabel) WHERE n.field IS NOT NULL RETURN n'
    );
""")

# Step 2: query the vector index on the subset
run_query("""
    CALL QUERY_VECTOR_INDEX('my_subset', 'field_index', $qv, 5)
    RETURN node.name, distance ORDER BY distance;
""", params={"qv": query_vector})
```

### Rainbow KG Schema

```
ReviewedOpportunity
  -[:REVIEWS]->                 Opportunity       (name_embedding ✦)
  -[:HAS_RISK]->                RiskAnalysis      (risk_description_embedding ✦)
  -[:HAS_TECHNICAL_APPROACH]->  TechnicalApproach (architecture_embedding ✦, technical_stack_embedding ✦)
  -[:HAS_COMPETITOR]->          Competitor
  -[:HAS_PARTNER]->             Partner
  -[:HAS_TEAM_MEMBER]->         Person
  -[:DELIVERED_IN]->            Geo
Opportunity
  -[:HAS_CUSTOMER]->            Customer          (name_embedding ✦)
  -[:HAS_CONTACT]->             Person

✦ = HNSW vector index available (CALL QUERY_VECTOR_INDEX)
```
